# 附篇：用 faker 批量生成测试数据

---

这个脚本和前面 14 篇不太一样：它**不分析数据，而是造数据**。

为什么需要这个？因为学 pandas 最尴尬的事是**没有数据可练**：
真实数据往往涉密、拿不到；手写几百行又太累；
网上随便下载的数据，字段含义不明，出错了都不知道对不对。

所以学会**自己造一份"像真的"测试数据**，是很实用的技能。

## 学习目标

1. 会用 `faker` 生成姓名、电话、邮箱等"看起来真实"的假数据
2. 明白**固定随机种子**对"结果可复现"有多重要（以及原稿为什么没做到）
3. 会做基本的数据质量检查（字段、缺失值、重复值、取值范围）
4. 知道写文件时的两个坑：**覆盖原文件**和**电话号码当数字**


In [ ]:
import os
import random

import numpy as np
import pandas as pd
from faker import Faker

print('pandas 版本：', pd.__version__)
print('faker  版本：', Faker.__module__ and __import__('faker').VERSION)

OUT_DIR = os.path.join(os.getcwd(), '生成文件')
os.makedirs(OUT_DIR, exist_ok=True)
print('输出目录：', OUT_DIR)


---
# 一、`faker` 是什么

`faker` 是一个"**假数据生成器**"：你说要什么，它给你编一个。

| 想要 | 调用 | 示例输出 |
|---|---|---|
| 中文名（名） | `fake.first_name()` | `凤英` |
| 中文名（姓） | `fake.last_name()` | `杨` |
| 邮箱 | `fake.email()` | `changgang@example.org` |
| 手机号 | `fake.phone_number()` | `15133218196` |
| 日期 | `fake.date_between(...)` | `2025-03-10` |
| 公司名 | `fake.company()` | `某某科技有限公司` |

关键点：**它可以指定语言**。`Faker('zh_CN')` 生成的就是中文姓名、中文公司名。

> ⚠️ 注意：faker 生成的是**"看起来像真的"的假数据**，
> 里面的手机号、邮箱**都是无效的**，不要拿去打电话或发邮件。


In [ ]:
fake = Faker('zh_CN')

print('姓名：', fake.last_name() + fake.first_name())
print('邮箱：', fake.email())
print('手机：', fake.phone_number())
print('公司：', fake.company())


---
# 二、让"随机"变得"可复现"

原稿有一个**不太明显但很重要**的问题：它没有固定随机种子。

后果是：**每次运行生成的数据都不一样**。
这在调试时非常痛苦 —— 你今天发现"某行数据不对"，
明天重新跑一遍，数据全变了，根本没法复现问题。

解法：**固定随机种子**（`seed`）。同一个种子，永远生成同一批数据。


In [ ]:
def make_fake(seed=42):
    """每个被 faker 依赖的随机源都要一起固定，否则还是会变。"""
    Faker.seed(seed)        # faker 自己的随机源
    random.seed(seed)       # Python 内置 random
    np.random.seed(seed)    # numpy（下面给薪资加点波动会用到）
    return Faker('zh_CN')


# 验证：同样的种子，两次生成的结果完全一致
f1 = make_fake(42)
names1 = [f1.first_name() for _ in range(5)]

f2 = make_fake(42)
names2 = [f2.first_name() for _ in range(5)]

print('第 1 次：', names1)
print('第 2 次：', names2)
print('两次一样吗？', names1 == names2)


In [ ]:
# 对比：不固定种子，结果就不一样
f3 = Faker('zh_CN')
print('不打种子：', [f3.first_name() for _ in range(5)])
f4 = Faker('zh_CN')
print('再来一次：', [f4.first_name() for _ in range(5)])
print('-> 完全不同。这还怎么排查问题？')


---
# 三、生成 500 条员工数据

字段设计：

| 字段 | 类型 | 说明 | 怎么造 |
|---|---|---|---|
| `employ_id` | 字符串 | 工号 | `EMP0001` 按序号生成 |
| `first_name` | 字符串 | 名 | `fake.first_name()` |
| `last_name` | 字符串 | 姓 | `fake.last_name()` |
| `email` | 字符串 | 公司邮箱 | `fake.email()` |
| `phone_number` | **字符串** | 手机号 | `fake.phone_number()` |
| `job_name` | 字符串 | 职位 | 从职位表里随机抽 |
| `salarys` | 整数 | 月薪 | 按职位对应的薪资区间随机 |
| `age` | 整数 | 年龄 | 22~60 随机 |


In [ ]:
# 职位和对应的薪资区间（下限, 上限）
JOBS = {
    '软件工程师': (8000, 25000),
    '数据分析师': (7000, 20000),
    '产品经理': (10000, 30000),
    '销售专员': (5000, 15000),
    '市场营销': (6000, 18000),
    '人力资源': (5500, 16000),
    '财务专员': (6000, 17000),
    '客服代表': (4000, 12000),
    '项目经理': (12000, 35000),
    'UI设计师': (8000, 22000),
    '运维工程师': (7000, 20000),
    '测试工程师': (6000, 18000),
}
print('共', len(JOBS), '个职位')
for job, (lo, hi) in list(JOBS.items())[:4]:
    print(f'  {job}: {lo} ~ {hi}')


In [ ]:
def generate_employees(n=500, seed=42):
    """生成 n 条员工数据，返回 DataFrame。"""
    fake = make_fake(seed)
    rows = []
    for i in range(1, n + 1):
        job_name = random.choice(list(JOBS.keys()))
        salary_lo, salary_hi = JOBS[job_name]

        rows.append({
            'employ_id': f'EMP{i:04d}',                      # EMP0001 这种定长编号，排序方便
            'first_name': fake.first_name(),
            'last_name': fake.last_name(),
            'email': fake.email(),                           # faker 生成的是【拼音】邮箱，不是中文
            'phone_number': fake.phone_number(),             # 【字符串】不要当数字
            'job_name': job_name,
            'salarys': random.randint(salary_lo, salary_hi),
            'age': random.randint(22, 60),
        })
    return pd.DataFrame(rows)


df = generate_employees(500)
print('生成完成：', df.shape)
print(df.head(8))


> **原稿在这里有一个瑕疵**：它的邮箱是用中文名拼的 ——
>
> ```python
> email = f"{first_name.lower()}.{last_name.lower()}{i}@company.com"
> ```
>
> 生成出来就是 `凤英.杨1@company.com` 这种**含中文字符的邮箱**。
>
> 技术上虽然能存，但**真实邮件系统基本都会拒收**，
> 拿去注册、发信、对接接口全是问题。
> `fake.email()` 生成的是**拼音 ASCII 邮箱**（`changgang@example.org`），安全得多。

> 还有一个小点：原稿 `import numpy as np` 却没有用到它。
> 顺手删掉（或者在有需要时再用）—— 保持代码干净是好习惯。


## 3.1 邮箱可能重复：查出并修掉

假数据也得讲质量。`fake.email()` 生成的邮箱**有可能撞车**，
而邮箱在业务上通常要求唯一。所以生成后要检查一遍：


In [ ]:
dup_count = int(df['email'].duplicated().sum())
print('重复的邮箱数量：', dup_count)
print('员工编号唯一吗？', df['employ_id'].is_unique)

if dup_count:
    dup_mask = df['email'].duplicated(keep=False)          # keep=False 会把重复的都标出来
    print('\n重复的邮箱：')
    print(df.loc[dup_mask, ['employ_id', 'email']])


In [ ]:
# 【修复】邮箱重复的，改成用工号生成一个必定唯一的邮箱
df = df.copy()                                   # 【好习惯】先复制，别改原表
dup_mask = df['email'].duplicated(keep=False)
df.loc[dup_mask, 'email'] = df.loc[dup_mask, 'employ_id'].str.lower() + '@company.com'

print('修复后还有重复吗？', df['email'].duplicated().any())
print('员工编号唯一吗？  ', df['employ_id'].is_unique)


---
# 四、写文件：先说风险

原稿直接写 `employee_data.csv`，**会覆盖掉目录里已有的同名文件**。
目录里那份 `employee_data.csv` 是**其他 notebook 都在用的数据源**，
一旦覆盖，前面 pandas 各篇的示例结果就全变了。

所以这里默认**写到 `生成文件/` 下面**，不动原文件。
如果你确实想重新生成项目里的那份数据，把下面的开关打开。


In [ ]:
OVERWRITE_ORIGINAL = False        # ⚠️ 改成 True 才会覆盖项目里的 employee_data.csv

gen_path = os.path.join(OUT_DIR, 'employee_data_生成版.csv')

if OVERWRITE_ORIGINAL:
    gen_path = os.path.join(os.getcwd(), 'employee_data.csv')   # 覆盖原文件
    print('⚠️ 将覆盖：', gen_path)
else:
    print('安全模式，只写到这里：', gen_path)

# 【重要】phone_number 要按【字符串】写出去，避免读取时被当数字
df.to_csv(gen_path, index=False, encoding='utf-8-sig')
print('已写入，大小：', os.path.getsize(gen_path), '字节')


In [ ]:
# 读回来看看（注意 dtype 显式声明 phone_number 为字符串）
back = pd.read_csv(gen_path, encoding='utf-8-sig', dtype={'phone_number': str})
print('读回来形状：', back.shape)
print(back.head(5))

print('\n各列类型：')
print(back.dtypes)


---
# 五、数据质量检查

造完数据**不能直接就用**，至少要过一遍这几项：

| 检查项 | 怎么做 | 期望 |
|---|---|---|
| 行数列数对不对 | `df.shape` | 500 × 8 |
| 有没有空值 | `df.isna().sum()` | 全 0 |
| 编号/邮箱唯不唯一 | `is_unique` / `duplicated()` | 唯一 |
| 取值在不在合理区间 | `describe()` | 年龄 22~60；薪资在区间内 |
| 分类字段分布合不合理 | `value_counts()` | 别出现"某个职位 0 人" |


In [ ]:
print('=== ① 形状 ===')
print(df.shape)

print('\n=== ② 空值检查 ===')
print(df.isna().sum())


In [ ]:
print('=== ③ 唯一性 ===')
print('employ_id 唯一：', df['employ_id'].is_unique)
print('email     唯一：', df['email'].is_unique)


In [ ]:
print('=== ④ 数值范围 ===')
print(df[['salarys', 'age']].describe())


In [ ]:
print('=== ⑤ 职位分布 ===')
vc = df['job_name'].value_counts()
print(vc)
print('\n最多：', vc.index[0], vc.iloc[0], '人 | 最少：', vc.index[-1], vc.iloc[-1], '人')


In [ ]:
# 更严谨：校验每行薪资是否真的落在它职位的区间里
def salary_in_range(row):
    lo, hi = JOBS[row['job_name']]
    return lo <= row['salarys'] <= hi


bad = df[~df.apply(salary_in_range, axis=1)]
print('薪资超出职位区间的记录数：', len(bad), '（应为 0 ✅）')


---
# 六、顺手做个小分析

数据造出来了，正好练一下第 7 篇学的 `groupby`。


In [ ]:
summary = (
    df.groupby('job_name', as_index=False)
      .agg(人数=('employ_id', 'size'),
           平均薪资=('salarys', 'mean'),
           最高薪资=('salarys', 'max'),
           平均年龄=('age', 'mean'))
      .sort_values('平均薪资', ascending=False)
      .round(1)
)
print('各职位薪资情况（按平均薪资降序）：')
print(summary)


In [ ]:
out_csv = os.path.join(OUT_DIR, '职位薪资统计.csv')
summary.to_csv(out_csv, index=False, encoding='utf-8-sig')
print('已导出：', out_csv)


In [ ]:
print('=== 年龄分布（每 10 岁一档）===')
print(pd.cut(df['age'], bins=[20, 30, 40, 50, 60], labels=['21-30', '31-40', '41-50', '51-60'])
        .value_counts()
        .sort_index())


---
# 七、让数据"更真实"（可选进阶）

上面这份数据是"横截面"的：每个人只有一个薪资值。
真实场景里还会有：

| 需求 | 做法 |
|---|---|
| 加入"入职时间"，看员工工龄 | `fake.date_between(start_date='-5y', end_date='today')` |
| 让薪资随工龄增长（而不是纯随机） | 薪资 = 基准 × (1 + 0.08 × 工龄) + 随机扰动 |
| 让某些职位人数更多（符合金字塔结构） | 用 `random.choices(weights=[...])` |
| 加入少量缺失值，练习数据清洗 | 随机把 2% 的值设成 `np.nan` |
| 加入重复行，练习去重 | `pd.concat([df, df.sample(5)])` |

下面演示两个最有用的：


In [ ]:
df_rich = df.copy()

# ① 加入入职时间（过去 5 年内随机）
fake2 = make_fake(7)
df_rich['入职时间'] = [fake2.date_between(start_date='-5y', end_date='today')
                       for _ in range(len(df_rich))]

# 把日期转成真正的日期类型（faker 返回的是 datetime.date 对象）
df_rich['入职时间'] = pd.to_datetime(df_rich['入职时间'])

# ② 让薪资和工龄挂钩：今天 - 入职时间 = 工龄（年）
df_rich['工龄'] = ((pd.Timestamp.today() - df_rich['入职时间']).dt.days / 365).round(1)
df_rich['薪资_按工龄修正'] = (df_rich['salarys'] * (1 + 0.03 * df_rich['工龄'])).round(0).astype(int)

print(df_rich[['employ_id', 'job_name', '入职时间', '工龄', 'salarys', '薪资_按工龄修正']].head(6))


In [ ]:
print('工龄 vs 薪资 的相关系数：',
      round(df_rich['工龄'].corr(df_rich['薪资_按工龄修正']), 3))

print('\n按工龄分组看平均薪资：')
print(df_rich.groupby(pd.cut(df_rich['工龄'], bins=5), observed=True)['salarys']
        .mean().round(0))


In [ ]:
# ③ 故意注入一点"脏数据"，专门用来练数据清洗
df_dirty = df.copy()

rng = np.random.default_rng(0)
dirty_idx = rng.choice(df_dirty.index, size=10, replace=False)
df_dirty.loc[dirty_idx[:4], 'salarys'] = np.nan          # 4 个缺失薪资
df_dirty.loc[dirty_idx[4:7], 'phone_number'] = ''        # 3 个空电话

df_dirty = pd.concat([df_dirty, df_dirty.sample(3, random_state=0)], ignore_index=True)  # 3 行重复

print('脏数据版本：')
print('  行数：', len(df_dirty), '(原 500 + 重复 3)')
print('  salarys 缺失：', int(df_dirty['salarys'].isna().sum()))
print('  phone_number 为空：', int((df_dirty['phone_number'] == '').sum()))
print('  重复的 employ_id：', int(df_dirty['employ_id'].duplicated().sum()))

dirty_path = os.path.join(OUT_DIR, 'employee_data_dirty.csv')
df_dirty.to_csv(dirty_path, index=False, encoding='utf-8-sig')
print('\n已导出供练习：', dirty_path)


---
# 常见坑清单

| 坑 | 现象 | 正解 |
|---|---|---|
| 没固定随机种子 | 每次数据都不同，问题无法复现 | `Faker.seed()` + `random.seed()` + `np.random.seed()` |
| 只 seed 了 faker | 结果还是变（`random` / `numpy` 没种子） | **三个随机源都要打种子** |
| 中文拼邮箱 | `凤英.杨1@company.com`，真实系统拒收 | 用 `fake.email()`（英文拼音） |
| 电话号当数字 | 前导 0 丢失、科学计数法 | 存成字符串，读时 `dtype={'phone_number': str}` |
| 直接覆盖原数据文件 | 别的脚本读到的数据全变了 | 输出到独立目录；覆盖前先备份 |
| 生成后不检查 | 邮箱重复、空值、越界都发现不了 | 跑一遍 `isna` / `duplicated` / `describe` |
| `inplace=True` 改原表 | 想重来还得重新生成 | 先 `.copy()` 再改 |
| 把 faker 数据当真数据 | 假手机号/邮箱被误用 | 记住它只是**测试数据** |

---
# 动手练习

1. **换个主题**：生成 300 条商品数据，字段为
   `商品ID / 名称 / 分类 / 进价 / 售价 / 库存 / 上架日期`，
   要求：售价 > 进价、库存非负、分类有 5~8 种。
2. **可复现性实验**：把 `generate_employees` 的 `seed` 参数分别传 1 和 2，
   验证两次数据不同；都传 42，验证完全相同。
3. **清洗自己的脏数据**：读 `employee_data_dirty.csv`，把薪资缺失值用**同职位均值**填充，
   电话为空的行删掉，重复行去重，最后报出清洗前后行数。
4. **金字塔结构**：用 `random.choices(weights=[...])` 让"客服代表"人数最多、
   "项目经理"最少，更接近真实公司的人员结构。
5. **加校验**：写一个 `validate(df)` 函数，一次性检查
   空值、唯一性、薪资范围、年龄范围，有问题就抛出 `ValueError` 并说明原因。

---
# 答案思路

**1. 商品数据**

```python
CATS = ['电子', '食品', '服饰', '家居', '图书', '运动']
fk = make_fake(1)
rows = []
for i in range(1, 301):
    cost = random.randint(5, 500)
    rows.append({
        '商品ID': f'SKU{i:04d}',
        '名称': f'{random.choice(CATS)}商品{i}',
        '分类': random.choice(CATS),
        '进价': cost,
        '售价': round(cost * random.uniform(1.2, 2.5), 2),   # 保证 > 进价
        '库存': random.randint(0, 999),
        '上架日期': fk.date_between(start_date='-1y', end_date='today'),
    })
goods = pd.DataFrame(rows)
assert (goods['售价'] > goods['进价']).all()
assert (goods['库存'] >= 0).all()
print(goods.head())
```

**2. 可复现性实验**

```python
a = generate_employees(50, seed=1)
b = generate_employees(50, seed=2)
print('不同种子，内容相同吗：', a.equals(b))       # False

c = generate_employees(50, seed=42)
d = generate_employees(50, seed=42)
print('相同种子，内容相同吗：', c.equals(d))       # True
```

**3. 清洗脏数据**

```python
d = pd.read_csv(os.path.join(OUT_DIR, 'employee_data_dirty.csv'),
                encoding='utf-8-sig', dtype={'phone_number': str})
print('清洗前：', d.shape)

# ① 用同职位均值填薪资
d['salarys'] = d.groupby('job_name')['salarys'].transform(lambda s: s.fillna(s.mean())).round(0)

# ② 删掉电话为空的行
d = d[d['phone_number'].fillna('') != '']

# ③ 去重
d = d.drop_duplicates(subset='employ_id')

print('清洗后：', d.shape)
print('还有缺失吗：', int(d['salarys'].isna().sum()))
```

**4. 金字塔结构**

```python
# 权重越大，被抽中的概率越高
weights = [3, 2, 2, 8, 4, 3, 3, 10, 1, 3, 3, 5]      # 客服最多(10)，项目经理最少(1)
job_name = random.choices(list(JOBS), weights=weights, k=1)[0]
```

**5. 校验函数**

```python
def validate(d):
    errs = []
    if d.isna().any().any():
        errs.append(f'存在空值：{d.columns[d.isna().any()].tolist()}')
    if not d['employ_id'].is_unique:
        errs.append('employ_id 有重复')
    for _, r in d.iterrows():
        lo, hi = JOBS[r['job_name']]
        if not (lo <= r['salarys'] <= hi):
            errs.append(f"{r['employ_id']} 薪资 {r['salarys']} 超出 {r['job_name']} 区间")
    if not d['age'].between(18, 65).all():
        errs.append('年龄存在不合理取值')
    if errs:
        raise ValueError('数据校验失败：\n  ' + '\n  '.join(errs))
    return True

print(validate(df))     # True
```
